# GQE — language scoring

**MSc Financial Technology dissertation · University of Birmingham**

Produces `LPS_scores.csv`: one Linguistic Positivity Score per company-year,
computed by running FinBERT over the BRSR section of each firm's annual report.
This is the stage that could not run in the analysis environment, because
FinBERT is downloaded from the Hugging Face model repository at load time.

Everything downstream of this notebook is in `GQE_Analysis.ipynb`, which starts
from the CSV output rather than the PDFs.

**Runtime:** roughly two minutes per filing on a free CPU instance, or about ten
seconds each on a T4 GPU. Runtime → Change runtime type → T4 GPU is worth doing
before running the full corpus.

**Corpus:** the full corpus is around 14 GB, which is more than a free Drive
account holds. The shared folder carries a subset sufficient to demonstrate the
pipeline; the full set is available from the author. The notebook scores
whatever PDFs it finds, so a subset runs without modification.

## 1. Setup

In [ ]:
# ── CONFIGURATION ────────────────────────────────────────────────────────────
# Shared read-only Drive folder holding the BRSR PDFs.
PDF_FOLDER_URL = "PASTE_YOUR_SHARED_PDF_FOLDER_LINK_HERE"

PDF_DIR    = "/content/brsr_pdfs"
OUTPUT_CSV = "/content/LPS_scores.csv"

import subprocess, sys, os
from pathlib import Path

def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

pipq("gdown", "pymupdf", "transformers", "torch", "pysentiment2")

import gdown
PDFS = Path(PDF_DIR)
if not PDFS.exists() or not any(PDFS.glob("*.pdf")):
    gdown.download_folder(PDF_FOLDER_URL, output=PDF_DIR, quiet=False, use_cookies=False)

pdfs = sorted(PDFS.glob("*.pdf"))
print(f"\n{len(pdfs)} PDFs found")
for p in pdfs[:10]:
    print("   ", p.name)
if len(pdfs) > 10:
    print(f"    ... and {len(pdfs) - 10} more")

## 2. Filename to company-year

Filenames carry the company and the year, but inconsistently, and several
contain misspellings that appear in the source files themselves — "graism" for
Grasim, "induslnd" for IndusInd, "vendanta" for Vedanta.

The mapping is a hand-written lookup table. Fuzzy string matching was tested and
rejected: it produced four confident and wrong pairings, including Colgate India
to Coal India and ICICI Lombard to ICICI Bank. Either would have merged one
firm's disclosure language onto another firm's operational data and left no
trace in the output. A filename that matches nothing is reported and skipped,
never forced.

In [ ]:
import re

ALIASES = {
    "abb": "ABB India", "acc": "ACC Ltd", "adani enterprise": "Adani Enterprises",
    "adani green": "Adani Green Energy", "adani ports": "Adani Ports",
    "airtel": "Bharti Airtel", "alkem": "Alkem Laboratories", "ambuja": "Ambuja Cements",
    "angel one": "Angel One", "apollo hospital": "Apollo Hospitals",
    "apollo tyre": "Apollo Tyres", "ashok leyland": "Ashok Leyland",
    "asianpaints": "Asian Paints", "aurobindo": "Aurobindo Pharma",
    "axis bank": "Axis Bank", "bajaj auto": "Bajaj Auto", "bajaj finance": "Bajaj Finance",
    "bajaj finserv": "Bajaj Finserv", "bank of baroda": "Bank of Baroda",
    "bharat petroleum": "Bharat Petroleum", "biocon": "Biocon", "bosch": "Bosch India",
    "britannia": "Britannia Industries", "bse": "BSE Ltd",
    "cdsl": "Central Depository Services", "cholamandalam": "Cholamandalam Finance",
    "cipla": "Cipla", "coal india": "Coal India", "coforge": "Coforge",
    "colgate": "Colgate-Palmolive India", "cummins": "Cummins India",
    "dabur": "Dabur India", "divis": "Divi's Laboratories",
    "dr.reddy": "Dr Reddy's Laboratories", "eicher": "Eicher Motors",
    "emami": "Emami Ltd", "escorts": "Escorts Kubota", "federal bank": "Federal Bank",
    "godrej consumer": "Godrej Consumer Products",
    "graism": "Grasim Industries",       # misspelling present in the source filename
    "grasim": "Grasim Industries", "havells": "Havells India", "hcl": "HCL Technologies",
    "hdfc": "HDFC Bank", "hdfc life": "HDFC Life Insurance",
    "hero motocorp": "Hero MotoCorp", "hindalco": "Hindalco Industries",
    "hindustan unilever": "Hindustan Unilever", "icici": "ICICI Bank",
    "icici lombard": "ICICI Lombard General Insurance", "idfc": "IDFC First Bank",
    "indus tower": "Indus Towers",
    "induslnd": "IndusInd Bank",         # misspelling
    "infosys": "Infosys", "ioc": "Indian Oil Corporation",
    "ioc brsr": "Indian Oil Corporation", "itc": "ITC Ltd",
    "jindal steel": "Jindal Steel & Power", "jsw steel": "JSW Steel",
    "kotak bank": "Kotak Mahindra Bank", "l&t": "Larsen & Toubro",
    "l&t tech": "L&T Technology Services", "lic life insurance": "Life Insurance Corporation",
    "ltimindtree": "LTIMindtree", "lupin": "Lupin", "mahindra": "Mahindra & Mahindra",
    "marico": "Marico", "maruti suzuki": "Maruti Suzuki", "mphasis": "Mphasis",
    "muthoot finance": "Muthoot Finance", "nalco": "National Aluminium Company",
    "nestle": "Nestle India", "nmdc": "NMDC Ltd", "ntpc": "NTPC Ltd",
    "ongc": "Oil & Natural Gas Corporation", "oracle": "Oracle Financial Services",
    "paytm": "Paytm (One 97 Communications)", "persistent": "Persistent Systems",
    "pidilite": "Pidilite Industries", "pnb": "Punjab National Bank",
    "policy bazaar": "PB Fintech (PolicyBazaar)", "powergrid": "Power Grid Corporation",
    "reliance": "Reliance Industries", "sail": "Steel Authority of India",
    "sbi": "State Bank of India", "sbi life": "SBI Life Insurance",
    "shree cement": "Shree Cement", "shriram": "Shriram Finance", "siemen": "Siemens India",
    "sun pharma": "Sun Pharmaceutical", "tata consumers": "Tata Consumer Products",
    "tata power": "Tata Power", "tata steel": "Tata Steel",
    "tcs": "Tata Consultancy Services", "tech mahindra": "Tech Mahindra",
    "titan": "Titan Company", "torrent": "Torrent Pharmaceuticals",
    "tvs motors": "TVS Motor Company", "ultratech cement": "UltraTech Cement",
    "vendanta": "Vedanta Ltd",           # misspelling
    "vedanta": "Vedanta Ltd", "vodafone idea": "Vodafone Idea", "wipro": "Wipro",
}

YEAR_RANGE_RE  = re.compile(r"(20\d{2})\s*-\s*(\d{2})")
YEAR_SINGLE_RE = re.compile(r"(20\d{2})")


def match_company(stem):
    """Canonical company name for a filename stem, or None if unmatched."""
    s = re.sub(r"20\d{2}\s*-\s*\d{2}|20\d{2}", "", stem.lower()).strip()
    s = re.sub(r"\s+", " ", re.sub(r"\bbrsr\b", "", s)).strip()
    if s in ALIASES:
        return ALIASES[s]
    compact = s.replace(".", "").replace("&", "and")
    for k, v in ALIASES.items():
        if compact == k.replace(".", "").replace("&", "and"):
            return v
    return None


def parse_fiscal_year(stem):
    """FY2022-23, FY2023-24, or None. A bare '2023' means the year ENDING 2023."""
    m = YEAR_RANGE_RE.search(stem)
    if m:
        return {"2022": "FY2022-23", "2023": "FY2023-24"}.get(m.group(1))
    nums = YEAR_SINGLE_RE.findall(stem)
    if not nums:
        return None
    return {"2023": "FY2022-23", "2024": "FY2023-24"}.get(nums[-1])


from collections import defaultdict
groups, unmatched = defaultdict(list), []
for p in pdfs:
    co, fy = match_company(p.stem), parse_fiscal_year(p.stem)
    (groups[(co, fy)].append(p.name) if co and fy else unmatched.append(p.name))

print(f"{len(groups)} company-year targets from {len(pdfs)} files")
if unmatched:
    print(f"\nunmatched filenames ({len(unmatched)}):")
    for u in unmatched:
        print("   ", u)
dupes = {k: v for k, v in groups.items() if len(v) > 1}
if dupes:
    print(f"\ncompany-years with more than one candidate file ({len(dupes)}):")
    for k, v in dupes.items():
        print("   ", k, "->", v)

## 3. Locating the BRSR inside the annual report

A BRSR is one section of a 300-page document. Scoring the whole report would
measure the chairman's letter and the notes to the accounts instead.

The extractor anchors on pages where "Essential Indicators" or "Leadership
Indicators" appears as a section header. That phrasing is mandated by the BRSR
form and nothing else in an annual report uses it. Anchor pages are clustered,
then the block is widened outward over adjacent pages carrying any BRSR signal,
which recovers the Section A preamble at the front and the tables at the back.

Four things this design handles that a simpler one does not:

- Reports write "Principle 3", "PRINCIPLE: 3" and "Principle - 3". Matching on
  whitespace alone scored those filings zero of nine principles.
- One filing says "PRINCIPAL 1" throughout. A separate pattern catches it,
  guarded so that "principal 1,00,000" in a finance report does not match.
- One firm's integrated ESG chapter mentions "leadership indicators" in running
  prose and is longer than its actual BRSR, so a loose match selected a 57-page
  chapter over the real 16-page filing.
- A two-page contents index lists all nine principles and contains no report,
  which is why candidate blocks are ranked by page span before principle count.

In [ ]:
import fitz  # PyMuPDF

PRINCIPLE_RE     = re.compile(r"\bprinciple\s*[:\-\u2013\u2014.]?\s*([1-9])\b", re.I)
PRINCIPAL_TYPO   = re.compile(r"\bprincipal\s*[:\-\u2013\u2014.]?\s*([1-9])(?![\d,]|\.\d)", re.I)
SECTION_RE       = re.compile(r"\bsection\s*[abc]\b", re.I)
INDICATOR_RE     = re.compile(r"\b(essential indicators|leadership indicators)\b", re.I)
INDICATOR_HDR_RE = re.compile(r"(?:essential|leadership)\s+indicators\s*[:\n]", re.I)
BRSR_TITLE_RE    = re.compile(r"business responsibility (?:and|&) sustainability report", re.I)

MAX_GAP, ESS_CLUSTER_GAP = 4, 12


def _page_signal(text):
    if not text:
        return False, set()
    pr = {int(m) for m in PRINCIPLE_RE.findall(text)} | {int(m) for m in PRINCIPAL_TYPO.findall(text)}
    return bool(pr or SECTION_RE.search(text) or INDICATOR_RE.search(text)
                or BRSR_TITLE_RE.search(text)), pr


def extract_brsr_section(pdf_path):
    flags, psets, texts = [], [], []
    with fitz.open(pdf_path) as doc:
        total = doc.page_count
        for page in doc:
            t = page.get_text() or ""
            texts.append(t)
            f, pr = _page_signal(t)
            flags.append(f); psets.append(pr)

    empty = {"text": "", "pages": None, "num_pages": 0,
             "principles_found": [], "total_pages": total, "ess_pages": 0}
    if not any(flags):
        return empty

    n = len(flags)
    ess_idx = [i for i, t in enumerate(texts) if INDICATOR_HDR_RE.search(t)]

    if ess_idx:                                  # preferred path
        clusters, cur = [], [ess_idx[0]]
        for i in ess_idx[1:]:
            if i - cur[-1] <= ESS_CLUSTER_GAP:
                cur.append(i)
            else:
                clusters.append(cur)
                cur = [i]
        clusters.append(cur)
        best = max(clusters, key=len)
        start, end = best[0], best[-1]
        while start > 0 and flags[start - 1]:    # widen back over the preamble
            start -= 1
        while end < n - 1 and flags[end + 1]:    # and forward over the tables
            end += 1
    else:                                        # fallback: merge flagged runs
        runs, i = [], 0
        while i < n:
            if not flags[i]:
                i += 1; continue
            s = e = i; j = i + 1
            while j < n:
                if flags[j]:
                    e = j; j += 1
                elif any(flags[j:min(j + MAX_GAP, n)]):
                    j += 1
                else:
                    break
            runs.append((s, e)); i = e + 1
        # span first, then principle count: a contents page lists all nine
        start, end = max(runs, key=lambda r: (r[1] - r[0],
                                              len(set().union(*psets[r[0]:r[1] + 1]))))

    return {"text": "\n".join(texts[start:end + 1]),
            "pages": (start, end), "num_pages": end - start + 1,
            "principles_found": sorted(set().union(*psets[start:end + 1])),
            "total_pages": total, "ess_pages": len(ess_idx)}

## 4. The scorers

FinBERT is the primary measure. ClimateBERT is a climate-domain comparator and
the Loughran-McDonald finance dictionary is a rule-based baseline that predates
transformers.

Text is split into 300-word chunks before scoring. BERT models truncate at 512
tokens, so feeding a 20,000-word section in whole would score the first page and
discard the rest. Averaging chunk scores puts a long filing and a short one on
the same scale.

Note that ClimateBERT does not classify positive against negative. Its three
classes are opportunity, neutral and risk. An earlier version of this code
looked for labels that did not exist and silently returned the same value for
every filing, so the scorer now resolves label names from the model's own
configuration and raises if one is missing.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pysentiment2 as ps

CHUNK_WORDS, BATCH_SIZE = 300, 8
_lm = ps.LM()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


def chunk_text(text, chunk_words=CHUNK_WORDS):
    w = text.split()
    return [" ".join(w[i:i + chunk_words]) for i in range(0, len(w), chunk_words)] if w else []


class TransformerScorer:
    def __init__(self, model_name, pos_label, neg_label):
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(DEVICE).eval()
        id2label = {k: v.lower() for k, v in self.model.config.id2label.items()}
        label2id = {v: k for k, v in id2label.items()}
        # fail loudly if the model does not have the classes we asked for
        for lab in (pos_label, neg_label):
            if lab not in label2id:
                raise ValueError(f"{model_name} has no label '{lab}'. It has {sorted(label2id)}")
        self.pos_idx, self.neg_idx = label2id[pos_label], label2id[neg_label]

    def score_chunks(self, chunks):
        if not chunks:
            return None
        diffs = []
        with torch.no_grad():
            for i in range(0, len(chunks), BATCH_SIZE):
                enc = self.tok(chunks[i:i + BATCH_SIZE], return_tensors="pt",
                               truncation=True, max_length=512, padding=True).to(DEVICE)
                probs = torch.softmax(self.model(**enc).logits, dim=-1)
                diffs.extend((probs[:, self.pos_idx] - probs[:, self.neg_idx]).tolist())
        return sum(diffs) / len(diffs) if diffs else None


def score_lm(text):
    """(positive - negative) / (positive + negative) over the whole block."""
    if not text or not text.strip():
        return None
    tokens = _lm.tokenize(text)
    if not tokens:
        return None
    s = _lm.get_score(tokens)
    pos, neg = s["Positive"], s["Negative"]
    return 0.0 if pos + neg == 0 else float((pos - neg) / (pos + neg))


finbert = TransformerScorer("ProsusAI/finbert", "positive", "negative")
print("FinBERT loaded")
try:
    climatebert = TransformerScorer("climatebert/distilroberta-base-climate-sentiment",
                                    "opportunity", "risk")
    print("ClimateBERT loaded")
except Exception as e:
    climatebert = None
    print("ClimateBERT unavailable, continuing with FinBERT and LM only:", e)

## 5. Scoring the corpus

Where more than one file maps to the same company-year, all candidates are
extracted and the richest is kept, ranked by indicator pages, then principle
count, then block length. Dropped files are named in the log.

One limitation this ranking cannot fix: it selects the best BRSR content
available, not the right company's BRSR. Two files in the corpus were
mislabelled, and in one case the wrong file kept winning precisely because it
contained more BRSR content than the correct filing. Both were caught by hand.
No quality heuristic detects a wrong-company file that happens to be a good
document.

In [ ]:
import time, csv

FIELDS = ["company", "year", "LPS_finbert", "LPS_climatebert", "LPS_lm",
          "brsr_pages", "principles_found", "ess_pages", "word_count", "source_file"]

rows = []
targets = sorted(groups.items(), key=lambda kv: (kv[0][0], kv[0][1]))

for idx, ((company, fy), candidates) in enumerate(targets, 1):
    best = None
    for fname in candidates:
        try:
            res = extract_brsr_section(str(PDFS / fname))
        except Exception as e:
            print(f"  extraction failed for {fname}: {e}"); continue
        key = (res["ess_pages"], len(res["principles_found"]), res["num_pages"])
        if best is None or key > best[0]:
            best = (key, fname, res)

    if best is None or not best[2]["text"].strip():
        print(f"[{idx}/{len(targets)}] {company} {fy}: no BRSR content found"); continue

    _, fname, res = best
    if len(candidates) > 1:
        print(f"  {company} {fy}: kept {fname}, dropped {[c for c in candidates if c != fname]}")

    chunks = chunk_text(res["text"])
    t0 = time.time()
    fb = finbert.score_chunks(chunks)
    cb = climatebert.score_chunks(chunks) if climatebert else None
    lm = score_lm(res["text"])

    rows.append({"company": company, "year": fy, "LPS_finbert": fb,
                 "LPS_climatebert": cb, "LPS_lm": lm,
                 "brsr_pages": res["num_pages"],
                 "principles_found": len(res["principles_found"]),
                 "ess_pages": res["ess_pages"],
                 "word_count": len(res["text"].split()), "source_file": fname})

    print(f"[{idx}/{len(targets)}] {company:<32} {fy}  "
          f"finbert={fb:+.4f}  {res['num_pages']:>3}pg  "
          f"{len(res['principles_found'])}/9  {len(chunks):>3} chunks  {time.time()-t0:.1f}s")

import pandas as pd
lps = pd.DataFrame(rows, columns=FIELDS)
lps.to_csv(OUTPUT_CSV, index=False)
print(f"\n{len(lps)} company-years scored -> {OUTPUT_CSV}")

## 6. Validation

Nothing is computed from these scores until the checks below have been read.

The quality tiers used throughout the dissertation are defined here. SOLID means
five or more indicator pages, at least eight of nine principles, and a block of
at least eight pages. PARTIAL means real BRSR content below one of those bars.
FAILED means no BRSR form content at all.

The cross-method check is the one that matters most. FinBERT should read ESG
prose less negatively than the Loughran-McDonald dictionary, which was built on
10-K risk language and treats words like "emissions" and "liability" as negative
regardless of context. If that direction did not hold for most filings, the
fault would be in the scoring rather than in the companies.

In [ ]:
solid = ((lps["ess_pages"] >= 5) & (lps["principles_found"] >= 8) & (lps["brsr_pages"] >= 8))

# Documented exception, affecting one filing in the full corpus: a BRSR that is
# substantive on every independent measure but misses the principle bar because
# the header regex undercounted. Specified on extraction quality, not on results.
exception = ((lps["ess_pages"] >= 10) & (lps["principles_found"] >= 7)
             & (lps["brsr_pages"] >= 15) & (lps["word_count"] >= 15000))
lps["solid"] = solid | exception

failed  = lps["ess_pages"] == 0
partial = ~lps["solid"] & ~failed
print(f"SOLID   {lps['solid'].sum():>4}")
print(f"PARTIAL {partial.sum():>4}")
print(f"FAILED  {failed.sum():>4}")
print(f"TOTAL   {len(lps):>4}")
if exception.sum():
    print(f"\nreadmitted by the documented exception: "
          f"{lps.loc[exception & ~solid, ['company','year']].to_dict('records')}")

fb = lps["LPS_finbert"].dropna()
print(f"\nFinBERT  n={len(fb)}  min={fb.min():+.4f}  max={fb.max():+.4f}  "
      f"mean={fb.mean():+.4f}  sd={fb.std(ddof=0):.4f}")
if fb.std(ddof=0) < 1e-6:
    print("  FLAG: near-zero spread. That is a scoring fault, not a finding.")
if (fb == 0.0).any():
    print(f"  FLAG: {(fb == 0.0).sum()} score(s) exactly zero, usually meaning empty text.")

both = lps.dropna(subset=["LPS_finbert", "LPS_lm"])
if len(both):
    share = (both["LPS_finbert"] > both["LPS_lm"]).mean()
    print(f"\nFinBERT > Loughran-McDonald in {share:.1%} of filings "
          f"(expected direction; mean gap {(both['LPS_finbert'] - both['LPS_lm']).mean():+.4f})")

print("\nfive most positive (SOLID only):")
display(lps[lps["solid"]].nlargest(5, "LPS_finbert")[
    ["company", "year", "LPS_finbert", "brsr_pages", "principles_found", "word_count"]])
print("five least positive (SOLID only):")
display(lps[lps["solid"]].nsmallest(5, "LPS_finbert")[
    ["company", "year", "LPS_finbert", "brsr_pages", "principles_found", "word_count"]])

## 7. Download the output

`LPS_scores.csv` is the input to `GQE_Analysis.ipynb`, together with the
operational tracker. Only SOLID rows enter the divergence score.

In [ ]:
try:
    from google.colab import files
    files.download(OUTPUT_CSV)
except Exception:
    print(f"Not running in Colab. Output is at {OUTPUT_CSV}")

---

Three company-years in the full corpus could not be recovered at all: Godrej
Consumer Products, ICICI Lombard and Oracle Financial Services, all FY2022-23.
NSE holds a filing record for each and stores null as the attachment, so no PDF
was ever lodged. XBRL exists for all three and was deliberately not used, since
its text nodes are schema identifiers rather than prose and sentiment over them
would not be comparable with the PDF-derived text in every other row.

State Bank of India FY2022-23 survives only as a four-page truncation and was
excluded. Nestlé India FY2022-23 does not exist as a period: the company moved
from a December to a March year-end through a fifteen-month year running January
2023 to March 2024.